# Orquestador PLANILLAS - panel mes a mes (Athena + awswrangler)
Replica tu pipeline de planillas parametrizado y con UNION automatico.

1. `HM_PAGOS_PLANILLAS` (transacciones, toda la historia) - 1 vez
2. `HM_UNIVERSO_MAESTRO_PLANILLAS` - 1 vez
3. Loop por mes: PIVOT (M1..M12) + `HM_BASE_MODELO_PLANILLAS_<periodo>`
4. UNION -> `HM_PAGO_PLANILLAS_TOTAL`

**Variables nuevas (nombres comerciales en espanol):** sueldo_promedio_empleado_3m/12m, meses_sin_pagar_planilla, flg_dejo_de_pagar_planilla_1m/3m, caida_desde_pico_empleados_12m, empleados_actual_vs_maximo_12m, caida_desde_pico_importe_12m, aceleracion_caida_empleados, flg_desplome_empleados.

In [ ]:
# 1) Instalacion
%pip install -q awswrangler

In [ ]:
# 2) Imports y CONFIG
import time
import awswrangler as wr

DATABASE       = "disc_comercial"
WORKGROUP      = "ibk-discovery-comercial-work-group"
S3_OUTPUT      = "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/athena-results/"
PERIODO_DESDE  = 202501      # primer mes objetivo a generar
PERIODO_HASTA  = 202602      # ultimo mes objetivo
HIST_INICIO    = 202301      # historia para transacciones / universo
HIST_FIN       = 202601      # tope de la tabla de transacciones

In [ ]:
# 3) Utilidades
def add_months(yyyymm, n):
    y, m = divmod(yyyymm, 100)
    idx = y*12 + (m-1) + n
    return (idx//12)*100 + (idx%12) + 1

def month_range(desde, hasta):
    out, p = [], desde
    while p <= hasta:
        out.append(p); p = add_months(p, 1)
    return out

def run_ddl(sql, label=""):
    qid = wr.athena.start_query_execution(sql=sql, database=DATABASE, workgroup=WORKGROUP, s3_output=S3_OUTPUT)
    res = wr.athena.wait_query(query_execution_id=qid)
    st = res["Status"]["State"]
    if st != "SUCCEEDED":
        raise RuntimeError("[" + label + "] " + st + ": " + str(res["Status"].get("StateChangeReason","")))
    print("  OK [" + label + "] qid=" + qid)
    return qid

def drop_table(name):
    run_ddl("DROP TABLE IF EXISTS " + DATABASE + "." + name, "drop " + name)

In [ ]:
# 4a) Template - TRANSACCIONES (HM_PAGOS_PLANILLAS)
SQL_TRANSACC = r"""
CREATE TABLE disc_comercial.HM_PAGOS_PLANILLAS
WITH ( 
    format = 'Parquet', 
    parquet_compression = 'SNAPPY', 
    partitioned_by = ARRAY['PERIODO_2']
)
AS (
SELECT 
    YEAR(A.Fecha_Proceso_Dt) * 100 + MONTH(A.Fecha_Proceso_Dt) AS PERIODO,
    Num_Identif_Empresa AS NRO_DOC_ORD,
    COUNT(DISTINCT A.CUC_Num_Benef) AS NRO_PLANILLAS,
    
    SUM(CASE 
        WHEN COALESCE(A.Importe_Abono_Soles, 0) > 0 
        THEN CAST(A.Importe_Abono_Soles AS DECIMAL(15,2))
        WHEN COALESCE(A.Importe_Abono_Dolares, 0) > 0 
        THEN CAST(A.Importe_Abono_Dolares * COALESCE(T.tipo_cambio, 0) AS DECIMAL(15,2))
        ELSE 0 
    END) AS IMP_PLANILLAS_SOLARIZADO,
    
    AVG(CASE 
        WHEN COALESCE(A.Importe_Abono_Dolares, 0) > 0 
        THEN T.tipo_cambio 
    END) AS TC_PROMEDIO_USD,
    
    CAST(YEAR(A.Fecha_Proceso_Dt) * 100 + MONTH(A.Fecha_Proceso_Dt) AS VARCHAR) AS PERIODO_2
--SELECT *
FROM e_perm_aws.t_FACT_VPC_PPA_TRANSACC_INOF A 

LEFT JOIN (
    SELECT fecha_Cambio, tipo_cambio
    FROM e_perm_aws.t_dim_tipo_cambio_rsk 
    WHERE Cod_Tipo_Cambio = '1002'
        AND Cod_Moneda_Origen = 'USD'
) T ON T.fecha_Cambio = A.Fecha_Proceso_Dt

WHERE A.FechInformacion_Dt = A.fecha_proceso_dt
    AND A.Cod_Rubro IN ('04', '06')
    AND A.Cod_Canal = 'BPI'
    AND A.Tipo_Abono_Orig NOT IN ('IBC')
    AND CAST(YEAR(A.fecha_proceso_dt) * 100 + MONTH(A.fecha_proceso_dt) AS VARCHAR) <= '{hist_fin}'
    AND CAST(YEAR(A.fecha_proceso_dt) * 100 + MONTH(A.fecha_proceso_dt) AS VARCHAR) >= '{hist_inicio}'

GROUP BY 
    YEAR(A.Fecha_Proceso_Dt) * 100 + MONTH(A.Fecha_Proceso_Dt),
    Num_Identif_Empresa
)
"""

In [ ]:
# 4b) Template - UNIVERSO MAESTRO
SQL_UNIVERSO = r"""
CREATE TABLE disc_comercial.HM_UNIVERSO_MAESTRO_PLANILLAS
WITH ( 
    format = 'Parquet', 
    parquet_compression = 'SNAPPY'
)
AS (
    -- Todos los clientes únicos que ALGUNA VEZ tuvieron planillas desde 202301
    SELECT DISTINCT 
        NRO_DOC_ORD
    FROM disc_comercial.HM_PAGOS_PLANILLAS
    WHERE PERIODO >= {hist_inicio} 
      AND PERIODO <= {hist_fin}
)
"""

In [ ]:
# 4c) Template - PIVOT (M1..M12)
SQL_PIVOT = r"""
CREATE TABLE disc_comercial.HM_BASE_MODELO_PLANILLAS_PIVOT
WITH ( 
    format = 'Parquet', 
    parquet_compression = 'SNAPPY', 
    partitioned_by = ARRAY['PERIODO_OBJETIVO']
)
AS (
WITH universo_objetivo AS (
    SELECT DISTINCT
        {periodo_objetivo} AS PERIODO_OBJ,  -- ← CAMBIAR AQUÍ EL PERIODO OBJETIVO
        NRO_DOC_ORD
    FROM disc_comercial.HM_UNIVERSO_MAESTRO_PLANILLAS
),

ventana_historica AS (
    SELECT 
        U.PERIODO_OBJ,
        U.NRO_DOC_ORD,
        H.PERIODO,
        COALESCE(H.NRO_PLANILLAS, 0) AS NRO_PLANILLAS,  
        COALESCE(H.IMP_PLANILLAS_SOLARIZADO, 0) AS IMP_PLANILLAS_SOLARIZADO,
        H.TC_PROMEDIO_USD,
        ROW_NUMBER() OVER (PARTITION BY U.NRO_DOC_ORD ORDER BY H.PERIODO DESC) AS MES_ORDEN
    FROM universo_objetivo U
    LEFT JOIN disc_comercial.HM_PAGOS_PLANILLAS H
        ON U.NRO_DOC_ORD = H.NRO_DOC_ORD
        AND H.PERIODO < U.PERIODO_OBJ
        AND H.PERIODO >= CAST(
            YEAR(DATE_ADD('month', -12, DATE_PARSE(CAST(U.PERIODO_OBJ AS VARCHAR), '%Y%m'))) * 100 + 
            MONTH(DATE_ADD('month', -12, DATE_PARSE(CAST(U.PERIODO_OBJ AS VARCHAR), '%Y%m')))
            AS INTEGER
        )
)

SELECT 
    U.PERIODO_OBJ,
    U.NRO_DOC_ORD,
    
    COALESCE(MAX(CASE WHEN V.MES_ORDEN = 1 THEN V.NRO_PLANILLAS END), 0) AS M1_NRO_PLANILLAS,
    COALESCE(MAX(CASE WHEN V.MES_ORDEN = 2 THEN V.NRO_PLANILLAS END), 0) AS M2_NRO_PLANILLAS,
    COALESCE(MAX(CASE WHEN V.MES_ORDEN = 3 THEN V.NRO_PLANILLAS END), 0) AS M3_NRO_PLANILLAS,
    COALESCE(MAX(CASE WHEN V.MES_ORDEN = 4 THEN V.NRO_PLANILLAS END), 0) AS M4_NRO_PLANILLAS,
    COALESCE(MAX(CASE WHEN V.MES_ORDEN = 5 THEN V.NRO_PLANILLAS END), 0) AS M5_NRO_PLANILLAS,
    COALESCE(MAX(CASE WHEN V.MES_ORDEN = 6 THEN V.NRO_PLANILLAS END), 0) AS M6_NRO_PLANILLAS,
    COALESCE(MAX(CASE WHEN V.MES_ORDEN = 7 THEN V.NRO_PLANILLAS END), 0) AS M7_NRO_PLANILLAS,
    COALESCE(MAX(CASE WHEN V.MES_ORDEN = 8 THEN V.NRO_PLANILLAS END), 0) AS M8_NRO_PLANILLAS,
    COALESCE(MAX(CASE WHEN V.MES_ORDEN = 9 THEN V.NRO_PLANILLAS END), 0) AS M9_NRO_PLANILLAS,
    COALESCE(MAX(CASE WHEN V.MES_ORDEN = 10 THEN V.NRO_PLANILLAS END), 0) AS M10_NRO_PLANILLAS,
    COALESCE(MAX(CASE WHEN V.MES_ORDEN = 11 THEN V.NRO_PLANILLAS END), 0) AS M11_NRO_PLANILLAS,
    COALESCE(MAX(CASE WHEN V.MES_ORDEN = 12 THEN V.NRO_PLANILLAS END), 0) AS M12_NRO_PLANILLAS,
    
    COALESCE(CAST(MAX(CASE WHEN V.MES_ORDEN = 1 THEN V.IMP_PLANILLAS_SOLARIZADO END) AS DECIMAL(15,2)), 0) AS M1_IMP_PLANILLAS_SOLARIZADO,
    COALESCE(CAST(MAX(CASE WHEN V.MES_ORDEN = 2 THEN V.IMP_PLANILLAS_SOLARIZADO END) AS DECIMAL(15,2)), 0) AS M2_IMP_PLANILLAS_SOLARIZADO,
    COALESCE(CAST(MAX(CASE WHEN V.MES_ORDEN = 3 THEN V.IMP_PLANILLAS_SOLARIZADO END) AS DECIMAL(15,2)), 0) AS M3_IMP_PLANILLAS_SOLARIZADO,
    COALESCE(CAST(MAX(CASE WHEN V.MES_ORDEN = 4 THEN V.IMP_PLANILLAS_SOLARIZADO END) AS DECIMAL(15,2)), 0) AS M4_IMP_PLANILLAS_SOLARIZADO,
    COALESCE(CAST(MAX(CASE WHEN V.MES_ORDEN = 5 THEN V.IMP_PLANILLAS_SOLARIZADO END) AS DECIMAL(15,2)), 0) AS M5_IMP_PLANILLAS_SOLARIZADO,
    COALESCE(CAST(MAX(CASE WHEN V.MES_ORDEN = 6 THEN V.IMP_PLANILLAS_SOLARIZADO END) AS DECIMAL(15,2)), 0) AS M6_IMP_PLANILLAS_SOLARIZADO,
    COALESCE(CAST(MAX(CASE WHEN V.MES_ORDEN = 7 THEN V.IMP_PLANILLAS_SOLARIZADO END) AS DECIMAL(15,2)), 0) AS M7_IMP_PLANILLAS_SOLARIZADO,
    COALESCE(CAST(MAX(CASE WHEN V.MES_ORDEN = 8 THEN V.IMP_PLANILLAS_SOLARIZADO END) AS DECIMAL(15,2)), 0) AS M8_IMP_PLANILLAS_SOLARIZADO,
    COALESCE(CAST(MAX(CASE WHEN V.MES_ORDEN = 9 THEN V.IMP_PLANILLAS_SOLARIZADO END) AS DECIMAL(15,2)), 0) AS M9_IMP_PLANILLAS_SOLARIZADO,
    COALESCE(CAST(MAX(CASE WHEN V.MES_ORDEN = 10 THEN V.IMP_PLANILLAS_SOLARIZADO END) AS DECIMAL(15,2)), 0) AS M10_IMP_PLANILLAS_SOLARIZADO,
    COALESCE(CAST(MAX(CASE WHEN V.MES_ORDEN = 11 THEN V.IMP_PLANILLAS_SOLARIZADO END) AS DECIMAL(15,2)), 0) AS M11_IMP_PLANILLAS_SOLARIZADO,
    COALESCE(CAST(MAX(CASE WHEN V.MES_ORDEN = 12 THEN V.IMP_PLANILLAS_SOLARIZADO END) AS DECIMAL(15,2)), 0) AS M12_IMP_PLANILLAS_SOLARIZADO,
    
    CAST(U.PERIODO_OBJ AS VARCHAR) AS PERIODO_OBJETIVO

FROM universo_objetivo U
LEFT JOIN ventana_historica V
    ON U.NRO_DOC_ORD = V.NRO_DOC_ORD
    AND U.PERIODO_OBJ = V.PERIODO_OBJ

GROUP BY 
    U.PERIODO_OBJ,
    U.NRO_DOC_ORD
)
"""

In [ ]:
# 4d) Template - FEATURES (con variables nuevas en espanol comercial)
SQL_FEATURES = r"""
CREATE TABLE disc_comercial.HM_BASE_MODELO_PLANILLAS_{periodo_objetivo}
WITH ( 
    format = 'Parquet', 
    parquet_compression = 'SNAPPY'
)
AS (
SELECT 
    PERIODO_OBJETIVO,
    NRO_DOC_ORD,
    

    -- ===== NUEVAS: desenganche / fuga de planilla (derivadas de M1..M12) =====
    CAST((M1_IMP_PLANILLAS_SOLARIZADO+M2_IMP_PLANILLAS_SOLARIZADO+M3_IMP_PLANILLAS_SOLARIZADO) / NULLIF(M1_NRO_PLANILLAS+M2_NRO_PLANILLAS+M3_NRO_PLANILLAS,0) AS DECIMAL(15,2))                      AS sueldo_promedio_empleado_3m,
    CAST((M1_IMP_PLANILLAS_SOLARIZADO+M2_IMP_PLANILLAS_SOLARIZADO+M3_IMP_PLANILLAS_SOLARIZADO+M4_IMP_PLANILLAS_SOLARIZADO+M5_IMP_PLANILLAS_SOLARIZADO+M6_IMP_PLANILLAS_SOLARIZADO+M7_IMP_PLANILLAS_SOLARIZADO+M8_IMP_PLANILLAS_SOLARIZADO+M9_IMP_PLANILLAS_SOLARIZADO+M10_IMP_PLANILLAS_SOLARIZADO+M11_IMP_PLANILLAS_SOLARIZADO+M12_IMP_PLANILLAS_SOLARIZADO) / NULLIF(M1_NRO_PLANILLAS+M2_NRO_PLANILLAS+M3_NRO_PLANILLAS+M4_NRO_PLANILLAS+M5_NRO_PLANILLAS+M6_NRO_PLANILLAS+M7_NRO_PLANILLAS+M8_NRO_PLANILLAS+M9_NRO_PLANILLAS+M10_NRO_PLANILLAS+M11_NRO_PLANILLAS+M12_NRO_PLANILLAS,0) AS DECIMAL(15,2))                      AS sueldo_promedio_empleado_12m,
    CASE WHEN M1_NRO_PLANILLAS > 0 THEN 0 WHEN M2_NRO_PLANILLAS > 0 THEN 1 WHEN M3_NRO_PLANILLAS > 0 THEN 2 WHEN M4_NRO_PLANILLAS > 0 THEN 3 WHEN M5_NRO_PLANILLAS > 0 THEN 4 WHEN M6_NRO_PLANILLAS > 0 THEN 5 WHEN M7_NRO_PLANILLAS > 0 THEN 6 WHEN M8_NRO_PLANILLAS > 0 THEN 7 WHEN M9_NRO_PLANILLAS > 0 THEN 8 WHEN M10_NRO_PLANILLAS > 0 THEN 9 WHEN M11_NRO_PLANILLAS > 0 THEN 10 WHEN M12_NRO_PLANILLAS > 0 THEN 11 ELSE 12 END                                                AS meses_sin_pagar_planilla,
    CASE WHEN M1_NRO_PLANILLAS=0 AND (M2_NRO_PLANILLAS+M3_NRO_PLANILLAS)>0 THEN 1 ELSE 0 END       AS flg_dejo_de_pagar_planilla_1m,
    CASE WHEN (M1_NRO_PLANILLAS+M2_NRO_PLANILLAS+M3_NRO_PLANILLAS)=0 AND (M4_NRO_PLANILLAS+M5_NRO_PLANILLAS+M6_NRO_PLANILLAS)>0 THEN 1 ELSE 0 END AS flg_dejo_de_pagar_planilla_3m,
    CAST((GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS, M7_NRO_PLANILLAS, M8_NRO_PLANILLAS, M9_NRO_PLANILLAS, M10_NRO_PLANILLAS, M11_NRO_PLANILLAS, M12_NRO_PLANILLAS) - M1_NRO_PLANILLAS) * 100.0 / NULLIF(GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS, M7_NRO_PLANILLAS, M8_NRO_PLANILLAS, M9_NRO_PLANILLAS, M10_NRO_PLANILLAS, M11_NRO_PLANILLAS, M12_NRO_PLANILLAS),0) AS DECIMAL(10,2)) AS caida_desde_pico_empleados_12m,
    CAST(M1_NRO_PLANILLAS * 100.0 / NULLIF(GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS, M7_NRO_PLANILLAS, M8_NRO_PLANILLAS, M9_NRO_PLANILLAS, M10_NRO_PLANILLAS, M11_NRO_PLANILLAS, M12_NRO_PLANILLAS),0) AS DECIMAL(10,2))            AS empleados_actual_vs_maximo_12m,
    CAST((GREATEST(M1_IMP_PLANILLAS_SOLARIZADO, M2_IMP_PLANILLAS_SOLARIZADO, M3_IMP_PLANILLAS_SOLARIZADO, M4_IMP_PLANILLAS_SOLARIZADO, M5_IMP_PLANILLAS_SOLARIZADO, M6_IMP_PLANILLAS_SOLARIZADO, M7_IMP_PLANILLAS_SOLARIZADO, M8_IMP_PLANILLAS_SOLARIZADO, M9_IMP_PLANILLAS_SOLARIZADO, M10_IMP_PLANILLAS_SOLARIZADO, M11_IMP_PLANILLAS_SOLARIZADO, M12_IMP_PLANILLAS_SOLARIZADO) - M1_IMP_PLANILLAS_SOLARIZADO) * 100.0 / NULLIF(GREATEST(M1_IMP_PLANILLAS_SOLARIZADO, M2_IMP_PLANILLAS_SOLARIZADO, M3_IMP_PLANILLAS_SOLARIZADO, M4_IMP_PLANILLAS_SOLARIZADO, M5_IMP_PLANILLAS_SOLARIZADO, M6_IMP_PLANILLAS_SOLARIZADO, M7_IMP_PLANILLAS_SOLARIZADO, M8_IMP_PLANILLAS_SOLARIZADO, M9_IMP_PLANILLAS_SOLARIZADO, M10_IMP_PLANILLAS_SOLARIZADO, M11_IMP_PLANILLAS_SOLARIZADO, M12_IMP_PLANILLAS_SOLARIZADO),0) AS DECIMAL(10,2)) AS caida_desde_pico_importe_12m,
    CAST((((M1_NRO_PLANILLAS+M2_NRO_PLANILLAS+M3_NRO_PLANILLAS)-(M4_NRO_PLANILLAS+M5_NRO_PLANILLAS+M6_NRO_PLANILLAS)) - ((M4_NRO_PLANILLAS+M5_NRO_PLANILLAS+M6_NRO_PLANILLAS)-(M7_NRO_PLANILLAS+M8_NRO_PLANILLAS+M9_NRO_PLANILLAS))) AS DECIMAL(15,2)) AS aceleracion_caida_empleados,
    CASE WHEN (M2_NRO_PLANILLAS+M3_NRO_PLANILLAS+M4_NRO_PLANILLAS)>0 AND M1_NRO_PLANILLAS < 0.5*((M2_NRO_PLANILLAS+M3_NRO_PLANILLAS+M4_NRO_PLANILLAS)/3.0) THEN 1 ELSE 0 END AS flg_desplome_empleados,
    -- ========================================
    -- FEATURES BÁSICAS - VALORES INDIVIDUALES POR MES
    -- ========================================
    M1_NRO_PLANILLAS,
    M2_NRO_PLANILLAS,
    M3_NRO_PLANILLAS,
    M4_NRO_PLANILLAS,
    M5_NRO_PLANILLAS,
    M6_NRO_PLANILLAS,
    M7_NRO_PLANILLAS,
    M8_NRO_PLANILLAS,
    M9_NRO_PLANILLAS,
    M10_NRO_PLANILLAS,
    M11_NRO_PLANILLAS,
    M12_NRO_PLANILLAS,
    
    M1_IMP_PLANILLAS_SOLARIZADO,
    M2_IMP_PLANILLAS_SOLARIZADO,
    M3_IMP_PLANILLAS_SOLARIZADO,
    M4_IMP_PLANILLAS_SOLARIZADO,
    M5_IMP_PLANILLAS_SOLARIZADO,
    M6_IMP_PLANILLAS_SOLARIZADO,
    M7_IMP_PLANILLAS_SOLARIZADO,
    M8_IMP_PLANILLAS_SOLARIZADO,
    M9_IMP_PLANILLAS_SOLARIZADO,
    M10_IMP_PLANILLAS_SOLARIZADO,
    M11_IMP_PLANILLAS_SOLARIZADO,
    M12_IMP_PLANILLAS_SOLARIZADO,
    
    -- ========================================
    -- FEATURES DE AGREGACIÓN - TOTALES
    -- ========================================
    M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS AS TOTAL_PLANILLAS_3M,
    M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS AS TOTAL_PLANILLAS_6M,
    M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS AS TOTAL_PLANILLAS_U12M,
    
    M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO AS TOTAL_IMP_3M,
    M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO AS TOTAL_IMP_U6M,
    M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO + M7_IMP_PLANILLAS_SOLARIZADO + M8_IMP_PLANILLAS_SOLARIZADO + M9_IMP_PLANILLAS_SOLARIZADO + M10_IMP_PLANILLAS_SOLARIZADO + M11_IMP_PLANILLAS_SOLARIZADO + M12_IMP_PLANILLAS_SOLARIZADO AS TOTAL_IMP_U12M,
    
    -- ========================================
    -- FEATURES DE AGREGACIÓN - PROMEDIOS
    -- ========================================
    CAST((M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) / 3.0 AS DECIMAL(15,2)) AS PROMEDIO_NRO_PLANILLAS_3M,
    CAST((M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) / 6.0 AS DECIMAL(15,2)) AS PROMEDIO_NRO_PLANILLAS_6M,
    CAST((M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) / 12.0 AS DECIMAL(15,2)) AS PROMEDIO_NRO_PLANILLAS_12M,
    
    CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO) / 3.0 AS DECIMAL(15,2)) AS PROMEDIO_IMP_3M,
    CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO) / 6.0 AS DECIMAL(15,2)) AS PROMEDIO_IMP_6M,
    CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO + M7_IMP_PLANILLAS_SOLARIZADO + M8_IMP_PLANILLAS_SOLARIZADO + M9_IMP_PLANILLAS_SOLARIZADO + M10_IMP_PLANILLAS_SOLARIZADO + M11_IMP_PLANILLAS_SOLARIZADO + M12_IMP_PLANILLAS_SOLARIZADO) / 12.0 AS DECIMAL(15,2)) AS PROMEDIO_IMP_12M,
    
    -- ========================================
    -- FEATURES DE AGREGACIÓN - MÁXIMOS Y MÍNIMOS
    -- ========================================
    GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS) AS MAX_NRO_PLANILLAS_3M,
    GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS) AS MAX_NRO_PLANILLAS_6M,
    GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS, M7_NRO_PLANILLAS, M8_NRO_PLANILLAS, M9_NRO_PLANILLAS, M10_NRO_PLANILLAS, M11_NRO_PLANILLAS, M12_NRO_PLANILLAS) AS MAX_NRO_PLANILLAS_12M,
    
    LEAST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS) AS MIN_NRO_PLANILLAS_3M,
    LEAST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS) AS MIN_NRO_PLANILLAS_6M,
    LEAST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS, M7_NRO_PLANILLAS, M8_NRO_PLANILLAS, M9_NRO_PLANILLAS, M10_NRO_PLANILLAS, M11_NRO_PLANILLAS, M12_NRO_PLANILLAS) AS MIN_NRO_PLANILLAS_12M,
    
    GREATEST(M1_IMP_PLANILLAS_SOLARIZADO, M2_IMP_PLANILLAS_SOLARIZADO, M3_IMP_PLANILLAS_SOLARIZADO) AS MAX_IMP_3M,
    GREATEST(M1_IMP_PLANILLAS_SOLARIZADO, M2_IMP_PLANILLAS_SOLARIZADO, M3_IMP_PLANILLAS_SOLARIZADO, M4_IMP_PLANILLAS_SOLARIZADO, M5_IMP_PLANILLAS_SOLARIZADO, M6_IMP_PLANILLAS_SOLARIZADO) AS MAX_IMP_6M,
    GREATEST(M1_IMP_PLANILLAS_SOLARIZADO, M2_IMP_PLANILLAS_SOLARIZADO, M3_IMP_PLANILLAS_SOLARIZADO, M4_IMP_PLANILLAS_SOLARIZADO, M5_IMP_PLANILLAS_SOLARIZADO, M6_IMP_PLANILLAS_SOLARIZADO, M7_IMP_PLANILLAS_SOLARIZADO, M8_IMP_PLANILLAS_SOLARIZADO, M9_IMP_PLANILLAS_SOLARIZADO, M10_IMP_PLANILLAS_SOLARIZADO, M11_IMP_PLANILLAS_SOLARIZADO, M12_IMP_PLANILLAS_SOLARIZADO) AS MAX_IMP_12M,
    
    -- ========================================
    -- FEATURES DE FRECUENCIA (ACTIVIDAD)
    -- ========================================
    CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END AS MESES_ACTIVOS_U3M,
    
    CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M4_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M5_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M6_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END AS MESES_ACTIVOS_U6M,
    
    CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M4_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M5_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M6_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M7_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M8_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M9_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M10_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M11_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
    CASE WHEN M12_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END AS MESES_ACTIVOS_U12M,
    
    CAST((CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END) * 100.0 / 3 AS DECIMAL(5,2)) AS PORC_MESES_ACTIVOS_U3M,
    
    CAST((CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M4_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M5_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M6_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END) * 100.0 / 6 AS DECIMAL(5,2)) AS PORC_MESES_ACTIVOS_U6M,
    
    CAST((CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M4_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M5_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M6_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M7_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M8_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M9_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M10_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M11_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
          CASE WHEN M12_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END) * 100.0 / 12 AS DECIMAL(5,2)) AS PORC_MESES_ACTIVOS_U12M,
    
    CASE WHEN (CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END) = 3 THEN 1 ELSE 0 END AS FLG_PAGO_TODOS_U3M,
    
    CASE WHEN (CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M4_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M5_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M6_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END) = 6 THEN 1 ELSE 0 END AS FLG_PAGO_TODOS_U6M,
    
    CASE WHEN (CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M4_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M5_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M6_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M7_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M8_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M9_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M10_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M11_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M12_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END) = 12 THEN 1 ELSE 0 END AS FLG_PAGO_TODOS_U12M,
    
    -- ========================================
    -- FEATURES DE CRECIMIENTO/DECRECIMIENTO
    -- ========================================
    CAST(M1_NRO_PLANILLAS - M2_NRO_PLANILLAS AS DECIMAL(15,2)) AS DIFF_PLANILLAS_M1_M2,
    CAST(M2_NRO_PLANILLAS - M3_NRO_PLANILLAS AS DECIMAL(15,2)) AS DIFF_PLANILLAS_M2_M3,
    CAST(M3_NRO_PLANILLAS - M4_NRO_PLANILLAS AS DECIMAL(15,2)) AS DIFF_PLANILLAS_M3_M4,
    
    COALESCE(
        CASE WHEN M2_NRO_PLANILLAS = 0 THEN NULL 
             ELSE CAST((M1_NRO_PLANILLAS - M2_NRO_PLANILLAS) * 100.0 / M2_NRO_PLANILLAS AS DECIMAL(10,2)) 
        END,
        0
    ) AS VAR_PORC_M1_M2,
    
    COALESCE(
        CASE WHEN M3_NRO_PLANILLAS = 0 THEN NULL 
             ELSE CAST((M2_NRO_PLANILLAS - M3_NRO_PLANILLAS) * 100.0 / M3_NRO_PLANILLAS AS DECIMAL(10,2)) 
        END,
        0
    ) AS VAR_PORC_M2_M3,
    
    COALESCE(
        CASE WHEN M4_NRO_PLANILLAS = 0 THEN NULL 
             ELSE CAST((M3_NRO_PLANILLAS - M4_NRO_PLANILLAS) * 100.0 / M4_NRO_PLANILLAS AS DECIMAL(10,2)) 
        END,
        0
    ) AS VAR_PORC_M3_M4,
    
    COALESCE(
        CASE WHEN (M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) = 0 THEN NULL
             ELSE CAST(((M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) - 
                        (M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS)) * 100.0 / 
                        (M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) AS DECIMAL(10,2))
        END,
        0
    ) AS VAR_PORC_3M_VS_3M,
    
    COALESCE(
        CASE WHEN (M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) = 0 THEN NULL
             ELSE CAST(((M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) - 
                        (M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS)) * 100.0 / 
                        (M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) AS DECIMAL(10,2))
        END,
        0
    ) AS VAR_PORC_6M_VS_6M,
    
    COALESCE(
        CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) = 0 THEN NULL
             ELSE CAST(((M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) / 3.0 - 
                        (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) / 12.0) * 100.0 / 
                       ((M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) / 12.0) AS DECIMAL(10,2))
        END,
        0
    ) AS VAR_PORC_3M_VS_PROM_12M,
    
    COALESCE(
        CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) = 0 THEN NULL
             ELSE CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO) / 
                       (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) AS DECIMAL(15,2))
        END,
        0
    ) AS TICKET_PROMEDIO_3M,
    
    COALESCE(
        CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) = 0 THEN NULL
             ELSE CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO) / 
                       (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) AS DECIMAL(15,2))
        END,
        0
    ) AS TICKET_PROMEDIO_6M,
    
    COALESCE(
        CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) = 0 THEN NULL
             ELSE CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO + M7_IMP_PLANILLAS_SOLARIZADO + M8_IMP_PLANILLAS_SOLARIZADO + M9_IMP_PLANILLAS_SOLARIZADO + M10_IMP_PLANILLAS_SOLARIZADO + M11_IMP_PLANILLAS_SOLARIZADO + M12_IMP_PLANILLAS_SOLARIZADO) / 
                       (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) AS DECIMAL(15,2))
        END,
        0
    ) AS TICKET_PROMEDIO_12M,
    
    CASE WHEN M1_NRO_PLANILLAS > M2_NRO_PLANILLAS AND M2_NRO_PLANILLAS > M3_NRO_PLANILLAS 
         THEN 1 ELSE 0 END AS FLAG_CRECIMIENTO_SOSTENIDO_3M,
    
    CASE WHEN M1_NRO_PLANILLAS < M2_NRO_PLANILLAS AND M2_NRO_PLANILLAS < M3_NRO_PLANILLAS 
         THEN 1 ELSE 0 END AS FLAG_DECRECIMIENTO_SOSTENIDO_3M,
    
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) / 3.0 > 
              (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) / 12.0
         THEN 1 ELSE 0 END AS FLAG_3M_SOBRE_PROMEDIO_12M,
    
    -- ========================================
    -- FEATURES DE RANGOS DE VOLUMEN
    -- ========================================
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) >= 1 THEN 1 ELSE 0 END AS FLAG_MAS_1_PLANILLA_3M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) >= 10 THEN 1 ELSE 0 END AS FLAG_MAS_10_PLANILLAS_3M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) >= 20 THEN 1 ELSE 0 END AS FLAG_MAS_20_PLANILLAS_3M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) >= 50 THEN 1 ELSE 0 END AS FLAG_MAS_50_PLANILLAS_3M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) >= 100 THEN 1 ELSE 0 END AS FLAG_MAS_100_PLANILLAS_3M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) >= 200 THEN 1 ELSE 0 END AS FLAG_MAS_200_PLANILLAS_3M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) >= 500 THEN 1 ELSE 0 END AS FLAG_MAS_500_PLANILLAS_3M,
    
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) >= 1 THEN 1 ELSE 0 END AS FLAG_MAS_1_PLANILLA_6M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) >= 10 THEN 1 ELSE 0 END AS FLAG_MAS_10_PLANILLAS_6M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) >= 20 THEN 1 ELSE 0 END AS FLAG_MAS_20_PLANILLAS_6M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) >= 50 THEN 1 ELSE 0 END AS FLAG_MAS_50_PLANILLAS_6M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) >= 100 THEN 1 ELSE 0 END AS FLAG_MAS_100_PLANILLAS_6M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) >= 200 THEN 1 ELSE 0 END AS FLAG_MAS_200_PLANILLAS_6M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) >= 500 THEN 1 ELSE 0 END AS FLAG_MAS_500_PLANILLAS_6M,
    
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) >= 1 THEN 1 ELSE 0 END AS FLAG_MAS_1_PLANILLA_12M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) >= 10 THEN 1 ELSE 0 END AS FLAG_MAS_10_PLANILLAS_12M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) >= 20 THEN 1 ELSE 0 END AS FLAG_MAS_20_PLANILLAS_12M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) >= 50 THEN 1 ELSE 0 END AS FLAG_MAS_50_PLANILLAS_12M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) >= 100 THEN 1 ELSE 0 END AS FLAG_MAS_100_PLANILLAS_12M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) >= 200 THEN 1 ELSE 0 END AS FLAG_MAS_200_PLANILLAS_12M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) >= 500 THEN 1 ELSE 0 END AS FLAG_MAS_500_PLANILLAS_12M,
    CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) >= 1000 THEN 1 ELSE 0 END AS FLAG_MAS_1000_PLANILLAS_12M,
    
    -- ========================================
    -- FEATURES DE RECENCIA
    -- ========================================
    CASE 
        WHEN M1_NRO_PLANILLAS > 0 THEN 0
        WHEN M2_NRO_PLANILLAS > 0 THEN 1
        WHEN M3_NRO_PLANILLAS > 0 THEN 2
        WHEN M4_NRO_PLANILLAS > 0 THEN 3
        WHEN M5_NRO_PLANILLAS > 0 THEN 4
        WHEN M6_NRO_PLANILLAS > 0 THEN 5
        WHEN M7_NRO_PLANILLAS > 0 THEN 6
        WHEN M8_NRO_PLANILLAS > 0 THEN 7
        WHEN M9_NRO_PLANILLAS > 0 THEN 8
        WHEN M10_NRO_PLANILLAS > 0 THEN 9
        WHEN M11_NRO_PLANILLAS > 0 THEN 10
        WHEN M12_NRO_PLANILLAS > 0 THEN 11
        ELSE 18
    END AS MESES_DESDE_ULTIMA_ACTIVIDAD,
    
    CASE WHEN (CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END) >= 1 
         THEN 1 ELSE 0 END AS FLAG_CLIENTE_ACTIVO_3M,
    
    CASE WHEN (CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END) = 0
         AND  (CASE WHEN M4_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M5_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M6_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END) >= 1
         THEN 1 ELSE 0 END AS FLAG_CLIENTE_EN_RIESGO,
    
    CASE WHEN (CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M4_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M5_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M6_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END) = 0
         AND  (CASE WHEN M7_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M8_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M9_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M10_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M11_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
               CASE WHEN M12_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END) >= 1
         THEN 1 ELSE 0 END AS FLAG_CLIENTE_PERDIDO,
    
    CASE 
        WHEN M1_NRO_PLANILLAS > 0 THEN M1_NRO_PLANILLAS
        WHEN M2_NRO_PLANILLAS > 0 THEN M2_NRO_PLANILLAS
        WHEN M3_NRO_PLANILLAS > 0 THEN M3_NRO_PLANILLAS
        WHEN M4_NRO_PLANILLAS > 0 THEN M4_NRO_PLANILLAS
        WHEN M5_NRO_PLANILLAS > 0 THEN M5_NRO_PLANILLAS
        WHEN M6_NRO_PLANILLAS > 0 THEN M6_NRO_PLANILLAS
        WHEN M7_NRO_PLANILLAS > 0 THEN M7_NRO_PLANILLAS
        WHEN M8_NRO_PLANILLAS > 0 THEN M8_NRO_PLANILLAS
        WHEN M9_NRO_PLANILLAS > 0 THEN M9_NRO_PLANILLAS
        WHEN M10_NRO_PLANILLAS > 0 THEN M10_NRO_PLANILLAS
        WHEN M11_NRO_PLANILLAS > 0 THEN M11_NRO_PLANILLAS
        WHEN M12_NRO_PLANILLAS > 0 THEN M12_NRO_PLANILLAS
        ELSE 0
    END AS PLANILLAS_ULTIMA_ACTIVIDAD,
    
    CASE 
        WHEN M1_NRO_PLANILLAS > 0 THEN M1_IMP_PLANILLAS_SOLARIZADO
        WHEN M2_NRO_PLANILLAS > 0 THEN M2_IMP_PLANILLAS_SOLARIZADO
        WHEN M3_NRO_PLANILLAS > 0 THEN M3_IMP_PLANILLAS_SOLARIZADO
        WHEN M4_NRO_PLANILLAS > 0 THEN M4_IMP_PLANILLAS_SOLARIZADO
        WHEN M5_NRO_PLANILLAS > 0 THEN M5_IMP_PLANILLAS_SOLARIZADO
        WHEN M6_NRO_PLANILLAS > 0 THEN M6_IMP_PLANILLAS_SOLARIZADO
        WHEN M7_NRO_PLANILLAS > 0 THEN M7_IMP_PLANILLAS_SOLARIZADO
        WHEN M8_NRO_PLANILLAS > 0 THEN M8_IMP_PLANILLAS_SOLARIZADO
        WHEN M9_NRO_PLANILLAS > 0 THEN M9_IMP_PLANILLAS_SOLARIZADO
        WHEN M10_NRO_PLANILLAS > 0 THEN M10_IMP_PLANILLAS_SOLARIZADO
        WHEN M11_NRO_PLANILLAS > 0 THEN M11_IMP_PLANILLAS_SOLARIZADO
        WHEN M12_NRO_PLANILLAS > 0 THEN M12_IMP_PLANILLAS_SOLARIZADO
        ELSE 0
    END AS IMP_ULTIMA_ACTIVIDAD,
    
    CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END AS FLAG_ACTIVO_M1,
    
    -- ========================================
    -- FEATURES DE CONCENTRACIÓN
    -- ========================================
    COALESCE(
        CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) = 0 THEN NULL
             ELSE CAST((M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) * 100.0 / 
                       (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) AS DECIMAL(5,2))
        END,
        0
    ) AS PORC_CONCENTRACION_3M_EN_12M,
    
    COALESCE(
        CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) = 0 THEN NULL
             ELSE CAST((M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) * 100.0 / 
                       (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) AS DECIMAL(5,2))
        END,
        0
    ) AS PORC_CONCENTRACION_6M_EN_12M,
    
    -- ========================================
    -- FLAGS BINARIOS POR UMBRAL DE CANTIDAD
    -- ========================================
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS) > 5 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_5_ALGUN_MES_3M,
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS) > 10 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_10_ALGUN_MES_3M,
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS) > 20 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_20_ALGUN_MES_3M,
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS) > 50 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_50_ALGUN_MES_3M,
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS) > 100 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_100_ALGUN_MES_3M,
    
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS) > 5 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_5_ALGUN_MES_6M,
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS) > 10 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_10_ALGUN_MES_6M,
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS) > 20 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_20_ALGUN_MES_6M,
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS) > 50 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_50_ALGUN_MES_6M,
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS) > 100 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_100_ALGUN_MES_6M,
    
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS, M7_NRO_PLANILLAS, M8_NRO_PLANILLAS, M9_NRO_PLANILLAS, M10_NRO_PLANILLAS, M11_NRO_PLANILLAS, M12_NRO_PLANILLAS) > 5 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_5_ALGUN_MES_12M,
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS, M7_NRO_PLANILLAS, M8_NRO_PLANILLAS, M9_NRO_PLANILLAS, M10_NRO_PLANILLAS, M11_NRO_PLANILLAS, M12_NRO_PLANILLAS) > 10 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_10_ALGUN_MES_12M,
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS, M7_NRO_PLANILLAS, M8_NRO_PLANILLAS, M9_NRO_PLANILLAS, M10_NRO_PLANILLAS, M11_NRO_PLANILLAS, M12_NRO_PLANILLAS) > 20 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_20_ALGUN_MES_12M,
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS, M7_NRO_PLANILLAS, M8_NRO_PLANILLAS, M9_NRO_PLANILLAS, M10_NRO_PLANILLAS, M11_NRO_PLANILLAS, M12_NRO_PLANILLAS) > 50 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_50_ALGUN_MES_12M,
    CASE WHEN GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS, M7_NRO_PLANILLAS, M8_NRO_PLANILLAS, M9_NRO_PLANILLAS, M10_NRO_PLANILLAS, M11_NRO_PLANILLAS, M12_NRO_PLANILLAS) > 100 THEN 1 ELSE 0 END AS FLG_TUVO_MAS_100_ALGUN_MES_12M,
    
    -- ========================================
    -- *** NUEVAS VARIABLES PARA MODELO >100K ***
    -- ========================================
    
    -- =============================================
    -- 1. UMBRALES DE FACTURACIÓN (CRÍTICO)
    -- =============================================
    CASE WHEN GREATEST(M1_IMP_PLANILLAS_SOLARIZADO, M2_IMP_PLANILLAS_SOLARIZADO, M3_IMP_PLANILLAS_SOLARIZADO, 
                       M4_IMP_PLANILLAS_SOLARIZADO, M5_IMP_PLANILLAS_SOLARIZADO, M6_IMP_PLANILLAS_SOLARIZADO,
                       M7_IMP_PLANILLAS_SOLARIZADO, M8_IMP_PLANILLAS_SOLARIZADO, M9_IMP_PLANILLAS_SOLARIZADO,
                       M10_IMP_PLANILLAS_SOLARIZADO, M11_IMP_PLANILLAS_SOLARIZADO, M12_IMP_PLANILLAS_SOLARIZADO) > 100000 
         THEN 1 ELSE 0 END AS FLG_NOMINA_MAYOR_100K_PLANILLAS_12M,
    
    CASE WHEN GREATEST(M1_IMP_PLANILLAS_SOLARIZADO, M2_IMP_PLANILLAS_SOLARIZADO, M3_IMP_PLANILLAS_SOLARIZADO, 
                       M4_IMP_PLANILLAS_SOLARIZADO, M5_IMP_PLANILLAS_SOLARIZADO, M6_IMP_PLANILLAS_SOLARIZADO,
                       M7_IMP_PLANILLAS_SOLARIZADO, M8_IMP_PLANILLAS_SOLARIZADO, M9_IMP_PLANILLAS_SOLARIZADO,
                       M10_IMP_PLANILLAS_SOLARIZADO, M11_IMP_PLANILLAS_SOLARIZADO, M12_IMP_PLANILLAS_SOLARIZADO) > 50000 
         THEN 1 ELSE 0 END AS FLG_NOMINA_MAYOR_50K_PLANILLAS_12M,
    
    (CASE WHEN M1_IMP_PLANILLAS_SOLARIZADO > 100000 THEN 1 ELSE 0 END +
     CASE WHEN M2_IMP_PLANILLAS_SOLARIZADO > 100000 THEN 1 ELSE 0 END +
     CASE WHEN M3_IMP_PLANILLAS_SOLARIZADO > 100000 THEN 1 ELSE 0 END +
     CASE WHEN M4_IMP_PLANILLAS_SOLARIZADO > 100000 THEN 1 ELSE 0 END +
     CASE WHEN M5_IMP_PLANILLAS_SOLARIZADO > 100000 THEN 1 ELSE 0 END +
     CASE WHEN M6_IMP_PLANILLAS_SOLARIZADO > 100000 THEN 1 ELSE 0 END +
     CASE WHEN M7_IMP_PLANILLAS_SOLARIZADO > 100000 THEN 1 ELSE 0 END +
     CASE WHEN M8_IMP_PLANILLAS_SOLARIZADO > 100000 THEN 1 ELSE 0 END +
     CASE WHEN M9_IMP_PLANILLAS_SOLARIZADO > 100000 THEN 1 ELSE 0 END +
     CASE WHEN M10_IMP_PLANILLAS_SOLARIZADO > 100000 THEN 1 ELSE 0 END +
     CASE WHEN M11_IMP_PLANILLAS_SOLARIZADO > 100000 THEN 1 ELSE 0 END +
     CASE WHEN M12_IMP_PLANILLAS_SOLARIZADO > 100000 THEN 1 ELSE 0 END) AS CNT_MESES_NOMINA_MAYOR_100K_PLANILLAS_12M,
    
    (CASE WHEN M1_IMP_PLANILLAS_SOLARIZADO > 50000 THEN 1 ELSE 0 END +
     CASE WHEN M2_IMP_PLANILLAS_SOLARIZADO > 50000 THEN 1 ELSE 0 END +
     CASE WHEN M3_IMP_PLANILLAS_SOLARIZADO > 50000 THEN 1 ELSE 0 END +
     CASE WHEN M4_IMP_PLANILLAS_SOLARIZADO > 50000 THEN 1 ELSE 0 END +
     CASE WHEN M5_IMP_PLANILLAS_SOLARIZADO > 50000 THEN 1 ELSE 0 END +
     CASE WHEN M6_IMP_PLANILLAS_SOLARIZADO > 50000 THEN 1 ELSE 0 END +
     CASE WHEN M7_IMP_PLANILLAS_SOLARIZADO > 50000 THEN 1 ELSE 0 END +
     CASE WHEN M8_IMP_PLANILLAS_SOLARIZADO > 50000 THEN 1 ELSE 0 END +
     CASE WHEN M9_IMP_PLANILLAS_SOLARIZADO > 50000 THEN 1 ELSE 0 END +
     CASE WHEN M10_IMP_PLANILLAS_SOLARIZADO > 50000 THEN 1 ELSE 0 END +
     CASE WHEN M11_IMP_PLANILLAS_SOLARIZADO > 50000 THEN 1 ELSE 0 END +
     CASE WHEN M12_IMP_PLANILLAS_SOLARIZADO > 50000 THEN 1 ELSE 0 END) AS CNT_MESES_NOMINA_MAYOR_50K_PLANILLAS_12M,
    
    -- =============================================
    -- 2. CAPACIDAD DE ENDEUDAMIENTO
    -- =============================================
    ROUND(CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + 
                M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO + 
                M7_IMP_PLANILLAS_SOLARIZADO + M8_IMP_PLANILLAS_SOLARIZADO + M9_IMP_PLANILLAS_SOLARIZADO + 
                M10_IMP_PLANILLAS_SOLARIZADO + M11_IMP_PLANILLAS_SOLARIZADO + M12_IMP_PLANILLAS_SOLARIZADO) / 12.0 AS DECIMAL(15,2)) * 0.30, 2) AS CAPACIDAD_CUOTA_MENSUAL_PLANILLAS,
    
    ROUND(CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + 
                M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO + 
                M7_IMP_PLANILLAS_SOLARIZADO + M8_IMP_PLANILLAS_SOLARIZADO + M9_IMP_PLANILLAS_SOLARIZADO + 
                M10_IMP_PLANILLAS_SOLARIZADO + M11_IMP_PLANILLAS_SOLARIZADO + M12_IMP_PLANILLAS_SOLARIZADO) / 12.0 AS DECIMAL(15,2)) * 0.30 * 36, 2) AS CAPACIDAD_CREDITO_ESTIMADA_PLANILLAS,
    
    CASE WHEN (CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + 
                     M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO + 
                     M7_IMP_PLANILLAS_SOLARIZADO + M8_IMP_PLANILLAS_SOLARIZADO + M9_IMP_PLANILLAS_SOLARIZADO + 
                     M10_IMP_PLANILLAS_SOLARIZADO + M11_IMP_PLANILLAS_SOLARIZADO + M12_IMP_PLANILLAS_SOLARIZADO) / 12.0 AS DECIMAL(15,2)) * 0.30 * 36) > 100000 
         THEN 1 ELSE 0 END AS FLG_CAPACIDAD_MAYOR_100K_PLANILLAS,
    
    ROUND(CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + 
                M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO + 
                M7_IMP_PLANILLAS_SOLARIZADO + M8_IMP_PLANILLAS_SOLARIZADO + M9_IMP_PLANILLAS_SOLARIZADO + 
                M10_IMP_PLANILLAS_SOLARIZADO + M11_IMP_PLANILLAS_SOLARIZADO + M12_IMP_PLANILLAS_SOLARIZADO) / 12.0 AS DECIMAL(15,2)), 2) AS FLUJO_CAJA_ESTIMADO_MENSUAL_PLANILLAS,
    
    -- =============================================
    -- 3. SALARIO PROMEDIO POR EMPLEADO
    -- =============================================
    COALESCE(
        CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + 
                   M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) > 0
             THEN ROUND(CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + 
                              M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO + 
                              M7_IMP_PLANILLAS_SOLARIZADO + M8_IMP_PLANILLAS_SOLARIZADO + M9_IMP_PLANILLAS_SOLARIZADO + 
                              M10_IMP_PLANILLAS_SOLARIZADO + M11_IMP_PLANILLAS_SOLARIZADO + M12_IMP_PLANILLAS_SOLARIZADO) AS DECIMAL(15,2)) / 
                         (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + 
                          M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS), 2)
             ELSE 0
        END, 
    0) AS SALARIO_PROMEDIO_POR_EMPLEADO_PLANILLAS_12M,
    
    CASE WHEN (
        CASE WHEN (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + 
                   M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) > 0
             THEN (M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + 
                   M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO + 
                   M7_IMP_PLANILLAS_SOLARIZADO + M8_IMP_PLANILLAS_SOLARIZADO + M9_IMP_PLANILLAS_SOLARIZADO + 
                   M10_IMP_PLANILLAS_SOLARIZADO + M11_IMP_PLANILLAS_SOLARIZADO + M12_IMP_PLANILLAS_SOLARIZADO) / 
                  (M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + 
                   M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS)
             ELSE 0
        END
    ) > 3000 THEN 1 ELSE 0 END AS FLG_SALARIOS_ALTOS_PLANILLAS,
    
    -- =============================================
    -- 4. CRECIMIENTO EMPLEADOS
    -- =============================================
    COALESCE(
        CASE WHEN (M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) > 0
             THEN ROUND(CAST(((M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS) - 
                              (M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS)) * 100.0 / 
                             (M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS) AS DECIMAL(10,2)), 2)
             ELSE 0
        END,
    0) AS CRECIMIENTO_PCT_EMPLEADOS_PLANILLAS_3M,
    
    CASE 
        WHEN (M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO) > 
             (M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO) THEN 1
        WHEN (M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO) < 
             (M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO) THEN -1
        ELSE 0
    END AS DIRECCION_TENDENCIA_NOMINA_PLANILLAS,
    
    -- =============================================
    -- 5. VOLATILIDAD (Simplificada con rango)
    -- =============================================
    COALESCE(
        CASE WHEN CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + 
                        M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO + 
                        M7_IMP_PLANILLAS_SOLARIZADO + M8_IMP_PLANILLAS_SOLARIZADO + M9_IMP_PLANILLAS_SOLARIZADO + 
                        M10_IMP_PLANILLAS_SOLARIZADO + M11_IMP_PLANILLAS_SOLARIZADO + M12_IMP_PLANILLAS_SOLARIZADO) / 12.0 AS DECIMAL(15,2)) > 0
             THEN ROUND(
                    (GREATEST(M1_IMP_PLANILLAS_SOLARIZADO, M2_IMP_PLANILLAS_SOLARIZADO, M3_IMP_PLANILLAS_SOLARIZADO, 
                              M4_IMP_PLANILLAS_SOLARIZADO, M5_IMP_PLANILLAS_SOLARIZADO, M6_IMP_PLANILLAS_SOLARIZADO,
                              M7_IMP_PLANILLAS_SOLARIZADO, M8_IMP_PLANILLAS_SOLARIZADO, M9_IMP_PLANILLAS_SOLARIZADO,
                              M10_IMP_PLANILLAS_SOLARIZADO, M11_IMP_PLANILLAS_SOLARIZADO, M12_IMP_PLANILLAS_SOLARIZADO) -
                     LEAST(M1_IMP_PLANILLAS_SOLARIZADO, M2_IMP_PLANILLAS_SOLARIZADO, M3_IMP_PLANILLAS_SOLARIZADO, 
                           M4_IMP_PLANILLAS_SOLARIZADO, M5_IMP_PLANILLAS_SOLARIZADO, M6_IMP_PLANILLAS_SOLARIZADO,
                           M7_IMP_PLANILLAS_SOLARIZADO, M8_IMP_PLANILLAS_SOLARIZADO, M9_IMP_PLANILLAS_SOLARIZADO,
                           M10_IMP_PLANILLAS_SOLARIZADO, M11_IMP_PLANILLAS_SOLARIZADO, M12_IMP_PLANILLAS_SOLARIZADO)) /
                    CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + 
                          M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO + 
                          M7_IMP_PLANILLAS_SOLARIZADO + M8_IMP_PLANILLAS_SOLARIZADO + M9_IMP_PLANILLAS_SOLARIZADO + 
                          M10_IMP_PLANILLAS_SOLARIZADO + M11_IMP_PLANILLAS_SOLARIZADO + M12_IMP_PLANILLAS_SOLARIZADO) / 12.0 AS DECIMAL(15,2))
                 , 4)
             ELSE 0
        END,
    0) AS COEF_VARIACION_MONTO_PLANILLAS_12M,
    
    COALESCE(
        CASE WHEN CAST((M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) / 12.0 AS DECIMAL(15,2)) > 0
             THEN ROUND((GREATEST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS, M7_NRO_PLANILLAS, M8_NRO_PLANILLAS, M9_NRO_PLANILLAS, M10_NRO_PLANILLAS, M11_NRO_PLANILLAS, M12_NRO_PLANILLAS) - LEAST(M1_NRO_PLANILLAS, M2_NRO_PLANILLAS, M3_NRO_PLANILLAS, M4_NRO_PLANILLAS, M5_NRO_PLANILLAS, M6_NRO_PLANILLAS, M7_NRO_PLANILLAS, M8_NRO_PLANILLAS, M9_NRO_PLANILLAS, M10_NRO_PLANILLAS, M11_NRO_PLANILLAS, M12_NRO_PLANILLAS)) / CAST((M1_NRO_PLANILLAS + M2_NRO_PLANILLAS + M3_NRO_PLANILLAS + M4_NRO_PLANILLAS + M5_NRO_PLANILLAS + M6_NRO_PLANILLAS + M7_NRO_PLANILLAS + M8_NRO_PLANILLAS + M9_NRO_PLANILLAS + M10_NRO_PLANILLAS + M11_NRO_PLANILLAS + M12_NRO_PLANILLAS) / 12.0 AS DECIMAL(15,2)), 2)
             ELSE 0
        END,
    0) AS VOLATILIDAD_EMPLEADOS_PLANILLAS_12M,
    
    -- =============================================
    -- 6. RATIO DE CONSISTENCIA
    -- =============================================
    ROUND(CAST((CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                CASE WHEN M4_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                CASE WHEN M5_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                CASE WHEN M6_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                CASE WHEN M7_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                CASE WHEN M8_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                CASE WHEN M9_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                CASE WHEN M10_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                CASE WHEN M11_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                CASE WHEN M12_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END) AS DECIMAL(5,2)) / 12.0, 2) AS RATIO_CONSISTENCIA_PLANILLAS_12M,
    
    -- =============================================
    -- 7. CAPACIDAD SOSTENIDA (promedio × consistencia)
    -- =============================================
    ROUND(CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + 
                M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO + 
                M7_IMP_PLANILLAS_SOLARIZADO + M8_IMP_PLANILLAS_SOLARIZADO + M9_IMP_PLANILLAS_SOLARIZADO + 
                M10_IMP_PLANILLAS_SOLARIZADO + M11_IMP_PLANILLAS_SOLARIZADO + M12_IMP_PLANILLAS_SOLARIZADO) / 12.0 AS DECIMAL(15,2)) * 
          (CAST((CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M4_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M5_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M6_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M7_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M8_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M9_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M10_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M11_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M12_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END) AS DECIMAL(5,2)) / 12.0), 2) AS CAPACIDAD_SOSTENIDA_NOMINA_PLANILLAS_12M,
    
    -- =============================================
    -- 8. CAPACIDAD POTENCIAL (máximo × consistencia)
    -- =============================================
    ROUND(GREATEST(M1_IMP_PLANILLAS_SOLARIZADO, M2_IMP_PLANILLAS_SOLARIZADO, M3_IMP_PLANILLAS_SOLARIZADO, 
                   M4_IMP_PLANILLAS_SOLARIZADO, M5_IMP_PLANILLAS_SOLARIZADO, M6_IMP_PLANILLAS_SOLARIZADO,
                   M7_IMP_PLANILLAS_SOLARIZADO, M8_IMP_PLANILLAS_SOLARIZADO, M9_IMP_PLANILLAS_SOLARIZADO,
                   M10_IMP_PLANILLAS_SOLARIZADO, M11_IMP_PLANILLAS_SOLARIZADO, M12_IMP_PLANILLAS_SOLARIZADO) * 
          (CAST((CASE WHEN M1_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M2_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M3_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M4_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M5_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M6_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M7_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M8_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M9_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M10_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M11_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END +
                 CASE WHEN M12_NRO_PLANILLAS > 0 THEN 1 ELSE 0 END) AS DECIMAL(5,2)) / 12.0), 2) AS CAPACIDAD_POTENCIAL_NOMINA_PLANILLAS_12M,
    
    -- =============================================
    -- 9. RATIO RECIENTE VS HISTÓRICO
    -- =============================================
    COALESCE(
        CASE WHEN CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + 
                        M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO + 
                        M7_IMP_PLANILLAS_SOLARIZADO + M8_IMP_PLANILLAS_SOLARIZADO + M9_IMP_PLANILLAS_SOLARIZADO + 
                        M10_IMP_PLANILLAS_SOLARIZADO + M11_IMP_PLANILLAS_SOLARIZADO + M12_IMP_PLANILLAS_SOLARIZADO) / 12.0 AS DECIMAL(15,2)) > 0
             THEN ROUND((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO) / 3.0 / 
                        CAST((M1_IMP_PLANILLAS_SOLARIZADO + M2_IMP_PLANILLAS_SOLARIZADO + M3_IMP_PLANILLAS_SOLARIZADO + 
                              M4_IMP_PLANILLAS_SOLARIZADO + M5_IMP_PLANILLAS_SOLARIZADO + M6_IMP_PLANILLAS_SOLARIZADO + 
                              M7_IMP_PLANILLAS_SOLARIZADO + M8_IMP_PLANILLAS_SOLARIZADO + M9_IMP_PLANILLAS_SOLARIZADO + 
                              M10_IMP_PLANILLAS_SOLARIZADO + M11_IMP_PLANILLAS_SOLARIZADO + M12_IMP_PLANILLAS_SOLARIZADO) / 12.0 AS DECIMAL(15,2)), 2)
             ELSE 0
        END,
    0) AS RATIO_RECIENTE_VS_HISTORICO_PLANILLAS

FROM disc_comercial.HM_BASE_MODELO_PLANILLAS_PIVOT
)
"""

In [ ]:
# 5) Pasos
def build_transacc():
    print("[1] Transacciones HM_PAGOS_PLANILLAS")
    drop_table("HM_PAGOS_PLANILLAS")
    run_ddl(SQL_TRANSACC.format(hist_inicio=HIST_INICIO, hist_fin=HIST_FIN), "transacc")

def build_universo():
    print("[2] Universo maestro")
    drop_table("HM_UNIVERSO_MAESTRO_PLANILLAS")
    run_ddl(SQL_UNIVERSO.format(hist_inicio=HIST_INICIO, hist_fin=PERIODO_HASTA), "universo")

def build_mes(periodo):
    print("[mes] " + str(periodo))
    drop_table("HM_BASE_MODELO_PLANILLAS_PIVOT")
    run_ddl(SQL_PIVOT.format(periodo_objetivo=periodo), "pivot " + str(periodo))
    drop_table("HM_BASE_MODELO_PLANILLAS_" + str(periodo))
    run_ddl(SQL_FEATURES.format(periodo_objetivo=periodo), "features " + str(periodo))

def build_total(periodos):
    print("[3] Union total")
    drop_table("HM_PAGO_PLANILLAS_TOTAL")
    unions = "\nUNION ALL\n".join("SELECT * FROM " + DATABASE + ".HM_BASE_MODELO_PLANILLAS_" + str(p) for p in periodos)
    sql = ("CREATE TABLE " + DATABASE + ".HM_PAGO_PLANILLAS_TOTAL\n"
           "WITH (format='Parquet', parquet_compression='SNAPPY') AS (\n" + unions + "\n)")
    run_ddl(sql, "total")

## Ejecutar

In [ ]:
# 6) Correr todo
periodos = month_range(PERIODO_DESDE, PERIODO_HASTA)
t0 = time.time()
build_transacc()
build_universo()
for p in periodos:
    build_mes(p)
build_total(periodos)
print("Listo en " + str(round((time.time()-t0)/60,1)) + " min. Tabla: " + DATABASE + ".HM_PAGO_PLANILLAS_TOTAL")